In [1]:
import os
import numpy as np
import pandas as pd

print("۱. در حال بارگذاری دیتاست...")

try:
    df = pd.read_csv("US_Accidents_March23.csv")

except FileNotFoundError:
    import kagglehub

    print("دیتاست پیدا نشد. در حال دانلود...")
    path = kagglehub.dataset_download("kerynhan/us-accidents-march23")
    print("Path:", path)

    csv_file = os.path.join(path, "US_Accidents_March23.csv")
    df = pd.read_csv(csv_file)

if "out" in df.columns:
    df = df.drop(columns=["out"])

print(df.head())


۱. در حال بارگذاری دیتاست...
    ID   Source  Severity           Start_Time             End_Time  \
0  A-1  Source2         3  2016-02-08 05:46:00  2016-02-08 11:00:00   
1  A-2  Source2         2  2016-02-08 06:07:59  2016-02-08 06:37:59   
2  A-3  Source2         2  2016-02-08 06:49:27  2016-02-08 07:19:27   
3  A-4  Source2         3  2016-02-08 07:23:34  2016-02-08 07:53:34   
4  A-5  Source2         2  2016-02-08 07:39:07  2016-02-08 08:09:07   

   Start_Lat  Start_Lng  End_Lat  End_Lng  Distance(mi)  ... Roundabout  \
0  39.865147 -84.058723      NaN      NaN          0.01  ...      False   
1  39.928059 -82.831184      NaN      NaN          0.01  ...      False   
2  39.063148 -84.032608      NaN      NaN          0.01  ...      False   
3  39.747753 -84.205582      NaN      NaN          0.01  ...      False   
4  39.627781 -84.188354      NaN      NaN          0.01  ...      False   

  Station   Stop Traffic_Calming Traffic_Signal Turning_Loop Sunrise_Sunset  \
0   False  Fal

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings('ignore')

print("🚀 [INNOVERSE GOLD CHAMPIONSHIP] شروع آموزش معماری چندمرحله‌ای (Level-2 Stacking)...")

# ۱. بارگیری بهینه داده‌ها
needed_cols = [
    'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng',
    'Distance(mi)', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Speed(mph)', 'Amenity', 'Bump', 'Crossing',
    'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station',
    'Stop', 'Traffic_Calming', 'Traffic_Signal', 'State'
]

df = pd.read_csv('US_Accidents_March23.csv', usecols=needed_cols)
print(f"📦 تعداد کل نمونه‌های اولیه: {len(df):,}")

# ۲. تمیزکاری و پردازش زمان
df['Start_Time'] = pd.to_datetime(df['Start_Time'], format='mixed', errors='coerce')
df['End_Time'] = pd.to_datetime(df['End_Time'], format='mixed', errors='coerce')
df = df.dropna(subset=['Start_Time', 'Start_Lat', 'Start_Lng', 'State']).reset_index(drop=True)

df['Hour'] = df['Start_Time'].dt.hour.astype(np.float32)
df['DayOfWeek'] = df['Start_Time'].dt.dayofweek.astype(np.float32)
df['Month'] = df['Start_Time'].dt.month.astype(np.float32)
df['Duration_Minutes'] = ((df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60.0).astype(np.float32)

df = df[(df['Duration_Minutes'] >= 0) & (df['Duration_Minutes'] <= 1440)].reset_index(drop=True)

# ۳. متغیر هدف ترکیبی (با نوع np.int32 برای جلوگیری از Overflow)
weather_hazard_temp = (
    (df['Humidity(%)'] / (df['Visibility(mi)'].replace(0, 0.01) + 0.1)) *
    (100 / (df['Temperature(F)'] + 60).replace(0, 0.01))
).astype(np.float32)

df['Risk_Class'] = (
    (df['Severity'] >= 3) | 
    ((weather_hazard_temp > 25.0) & (df['Severity'] >= 2)) | 
    (df['Duration_Minutes'] > 240.0)
).astype(np.int32)

del weather_hazard_temp

# ۴. مهندسی ویژگی‌های پیشرفته
df['Hour_Sin'] = np.sin(2 * np.pi * df['Hour'] / 24.0).astype(np.float32)
df['Hour_Cos'] = np.cos(2 * np.pi * df['Hour'] / 24.0).astype(np.float32)
df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12.0).astype(np.float32)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12.0).astype(np.float32)
df['DayOfWeek_Sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7.0).astype(np.float32)
df['DayOfWeek_Cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7.0).astype(np.float32)

df['Is_Rush_Hour'] = ((df['DayOfWeek'] < 5) & (df['Hour'].isin([7, 8, 9, 16, 17, 18]))).astype(np.float32)
df['Is_Weekend'] = (df['DayOfWeek'] >= 5).astype(np.float32)
df['Wind_Chill_Effect'] = (df['Temperature(F)'] - (df['Wind_Speed(mph)'] * 0.7)).astype(np.float32)
df['Log_Distance'] = np.log1p(df['Distance(mi)'].clip(lower=0)).astype(np.float32)

infrastructure_cols = [
    'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit',
    'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal'
]
for col in infrastructure_cols:
    df[col] = df[col].astype(np.float32)

df['Infrastructure_Risk_Score'] = df[infrastructure_cols].sum(axis=1).astype(np.float32)

# خوشه‌بندی جغرافیایی
print("🗺️ در حال خوشه‌بندی جغرافیایی...")
kmeans = MiniBatchKMeans(n_clusters=50, random_state=42, batch_size=8192, n_init='auto')
df['Geo_Cluster'] = kmeans.fit_predict(df[['Start_Lat', 'Start_Lng']]).astype(np.float32)

global_mean_all = float(df['Risk_Class'].mean())
state_map_global = df.groupby('State')['Risk_Class'].mean().to_dict()

features = [
    'Start_Lat', 'Start_Lng', 'Geo_Cluster', 'State_Historical_Risk', 'Log_Distance',
    'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos',
    'Is_Rush_Hour', 'Is_Weekend', 'Wind_Chill_Effect',
    'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)',
    'Infrastructure_Risk_Score'
] + infrastructure_cols

# ۵. اعتبارسنجی متقاطع و آموزش
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

df['State_Historical_Risk'] = 0.0
oof_xgb = np.zeros(len(df), dtype=np.float32)
oof_lgb = np.zeros(len(df), dtype=np.float32)
oof_cat = np.zeros(len(df), dtype=np.float32)

xgb_models, lgb_models, cat_models = [], [], []
y = df['Risk_Class'].values

print(f"\n⚡ شروع آموزش ۵ فولد روی {len(df):,} نمونه...")

for fold, (train_idx, val_idx) in enumerate(skf.split(df, y)):
    print(f"\n--- [Fold {fold + 1}/{N_SPLITS}] ---")
    
    fold_train_df = df.iloc[train_idx]
    state_map_fold = fold_train_df.groupby('State')['Risk_Class'].mean().to_dict()
    fold_global_mean = fold_train_df['Risk_Class'].mean()
    
    df.loc[train_idx, 'State_Historical_Risk'] = df.iloc[train_idx]['State'].map(state_map_fold).fillna(fold_global_mean).values
    df.loc[val_idx, 'State_Historical_Risk'] = df.iloc[val_idx]['State'].map(state_map_fold).fillna(fold_global_mean).values
    
    X_fold = df[features].fillna(df[features].median()).astype(np.float32)
    
    scaler_fold = RobustScaler()
    X_tr = scaler_fold.fit_transform(X_fold.iloc[train_idx])
    X_va = scaler_fold.transform(X_fold.iloc[val_idx])
    
    y_tr, y_va = y[train_idx], y[val_idx]
    
    # محاسبه ایمن بدون Overflow
    num_pos = int(np.sum(y_tr, dtype=np.int64))
    pos_weight = float((len(y_tr) - num_pos) / num_pos)
    
    # مدل ۱: XGBoost
    xgb = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.04, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=42, n_jobs=-1, tree_method='hist')
    xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = xgb.predict_proba(X_va)[:, 1]
    xgb_models.append(xgb)
    
    # مدل ۲: LightGBM
    lgb = LGBMClassifier(n_estimators=400, max_depth=6, num_leaves=31, learning_rate=0.04, subsample=0.8, scale_pos_weight=pos_weight, random_state=42, n_jobs=-1, verbose=-1)
    lgb.fit(X_tr, y_tr)
    oof_lgb[val_idx] = lgb.predict_proba(X_va)[:, 1]
    lgb_models.append(lgb)
    
    # مدل ۳: CatBoost
    cat = CatBoostClassifier(iterations=400, depth=6, learning_rate=0.04, scale_pos_weight=pos_weight, random_seed=42, verbose=0)
    cat.fit(X_tr, y_tr)
    oof_cat[val_idx] = cat.predict_proba(X_va)[:, 1]
    cat_models.append(cat)
    
    print(f"   ├─ XGB AUC: {roc_auc_score(y_va, oof_xgb[val_idx]):.4f} | LGB AUC: {roc_auc_score(y_va, oof_lgb[val_idx]):.4f} | CAT AUC: {roc_auc_score(y_va, oof_cat[val_idx]):.4f}")

# ۶. آموزش Meta-Learner (Stacking)
print("\n🧠 در حال آموزش مدل سطح دو (Meta-Learner Stacking)...")
X_meta_train = np.column_stack([oof_xgb, oof_lgb, oof_cat])
meta_learner = LogisticRegression(C=1.0, max_iter=1000)
meta_learner.fit(X_meta_train, y)

oof_meta_preds = meta_learner.predict_proba(X_meta_train)[:, 1]
meta_auc = roc_auc_score(y, oof_meta_preds)
print(f"🏆 [امتیاز کل Stacking OOF ROC-AUC]: {meta_auc:.5f}")

# ۷. کالیبراسیون و ذخیره‌سازی
print("🎯 در حال اعمال کالیبراسیون ایزوتونیک...")
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrated_oof = calibrator.fit_transform(oof_meta_preds, y)

precisions, recalls, thresholds = precision_recall_curve(y, calibrated_oof)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold = float(thresholds[np.argmax(f1_scores)])

final_scaler = RobustScaler()
final_X = df[features].fillna(df[features].median()).astype(np.float32)
final_scaler.fit(final_X)

gold_artifact = {
    'xgb_models': xgb_models,
    'lgb_models': lgb_models,
    'cat_models': cat_models,
    'kmeans': kmeans,
    'scaler': final_scaler,
    'meta_learner': meta_learner,
    'calibrator': calibrator,
    'features': features,
    'state_map_global': state_map_global,
    'global_mean_all': global_mean_all,
    'best_threshold': best_threshold
}

joblib.dump(gold_artifact, 'innoverse_gold_model.pkl')
print(f"\n✅ [موفقیت کامل] تمام ۱۵ مدل و سیستم Stacking بدون ارور ذخیره شدند! آستانه قطعی بهینه: {best_threshold:.4f}")

In [ ]:
import joblib
import numpy as np
import pandas as pd

print("📥 در حال بارگیری سیستم انسامبل و Stacking...")

# ۱. بارگیری تمام مدل‌ها و آرتیفکت‌های ذخیره‌شده
try:
    artifact = joblib.load('innoverse_gold_model.pkl')
    print("✅ مدل با موفقیت بارگیری شد.")
except FileNotFoundError:
    print("❌ خطای عدم یافتن فایل! ابتدا فایل train.py را اجرا کنید تا innoverse_gold_model.pkl ساخته شود.")
    exit()

xgb_models = artifact['xgb_models']
lgb_models = artifact['lgb_models']
cat_models = artifact['cat_models']
kmeans = artifact['kmeans']
scaler = artifact['scaler']
meta_learner = artifact['meta_learner']
calibrator = artifact['calibrator']
features = artifact['features']
state_map_global = artifact['state_map_global']
global_mean_all = artifact['global_mean_all']
best_threshold = artifact['best_threshold']

# ۲. تابع پیش‌پردازش و مهندسی ویژگی‌ها
def process_input(raw_df):
    df = raw_df.copy()
    
    # ویژگی‌های چرخه‌ای زمان
    df['Hour_Sin'] = np.sin(2 * np.pi * df['Hour'] / 24.0).astype(np.float32)
    df['Hour_Cos'] = np.cos(2 * np.pi * df['Hour'] / 24.0).astype(np.float32)
    df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12.0).astype(np.float32)
    df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12.0).astype(np.float32)
    df['DayOfWeek_Sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7.0).astype(np.float32)
    df['DayOfWeek_Cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7.0).astype(np.float32)
    
    # ویژگی‌های رفتاری و جوی
    df['Is_Rush_Hour'] = ((df['DayOfWeek'] < 5) & (df['Hour'].isin([7, 8, 9, 16, 17, 18]))).astype(np.float32)
    df['Is_Weekend'] = (df['DayOfWeek'] >= 5).astype(np.float32)
    df['Wind_Chill_Effect'] = (df['Temperature(F)'] - (df['Wind_Speed(mph)'] * 0.7)).astype(np.float32)
    
    # نگاشت ریسک ایالت
    df['State_Historical_Risk'] = df['State'].map(state_map_global).fillna(global_mean_all).astype(np.float32)
    
    # خوشه‌بندی جغرافیایی و لوگ فاصله
    df['Geo_Cluster'] = kmeans.predict(df[['Start_Lat', 'Start_Lng']]).astype(np.float32)
    df['Log_Distance'] = np.log1p(df['Distance(mi)'].clip(lower=0)).astype(np.float32)
    
    # زیرساخت‌های شهری
    infrastructure_cols = [
        'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit',
        'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal'
    ]
    for col in infrastructure_cols:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = df[col].astype(np.float32)
        
    df['Infrastructure_Risk_Score'] = df[infrastructure_cols].sum(axis=1).astype(np.float32)
    
    # اطمینان از وجود همه ویژگی‌ها و پر کردن مقادیر مفقودی
    X_out = df[features].fillna(df[features].median()).astype(np.float32)
    return X_out

# ۳. تابع اصلی ارزیابی و استنتاج
def predict_risk(raw_df):
    X_proc = process_input(raw_df)
    X_scaled = scaler.transform(X_proc)
    
    # مرحله اول (Level-1): دریافت پیش‌بینی از تمامی ۱۵ مدل
    pred_xgb = np.mean([model.predict_proba(X_scaled)[:, 1] for model in xgb_models], axis=0)
    pred_lgb = np.mean([model.predict_proba(X_scaled)[:, 1] for model in lgb_models], axis=0)
    pred_cat = np.mean([model.predict_proba(X_scaled)[:, 1] for model in cat_models], axis=0)
    
    # مرحله دوم (Level-2): ترکیب هوشمند با Meta-Learner
    X_meta_test = np.column_stack([pred_xgb, pred_lgb, pred_cat])
    raw_meta_probs = meta_learner.predict_proba(X_meta_test)[:, 1]
    
    # کالیبراسیون ایزوتونیک نهایی
    calibrated_probs = calibrator.transform(raw_meta_probs)
    predictions = (calibrated_probs >= best_threshold).astype(int)
    
    return calibrated_probs, predictions, pred_xgb, pred_lgb, pred_cat

# ==========================================
# ۴. تست روی سناریوهای نمونه
# ==========================================
test_cases = pd.DataFrame([
    { # سناریوی ۱: بسیار خطرناک (طوفان شدید شبانه در تقاطع شلوغ ایالت ایلینوی)
        'Start_Lat': 41.8781, 'Start_Lng': -87.6298, 'State': 'IL', 'Distance(mi)': 12.5,
        'Hour': 2, 'DayOfWeek': 4, 'Month': 1, 'Temperature(F)': -10.0, 'Humidity(%)': 98.0,
        'Pressure(in)': 28.10, 'Visibility(mi)': 0.1, 'Wind_Speed(mph)': 50.0,
        'Amenity': 0.0, 'Bump': 0.0, 'Crossing': 1.0, 'Give_Way': 0.0, 'Junction': 1.0,
        'No_Exit': 0.0, 'Railway': 0.0, 'Roundabout': 0.0, 'Station': 0.0, 'Stop': 0.0,
        'Traffic_Calming': 0.0, 'Traffic_Signal': 1.0
    },
    { # سناریوی ۲: بسیار امن (روز آفتابی و ایده‌آل در کالیفرنیا)
        'Start_Lat': 34.0522, 'Start_Lng': -118.2437, 'State': 'CA', 'Distance(mi)': 0.1,
        'Hour': 14, 'DayOfWeek': 1, 'Month': 6, 'Temperature(F)': 75.0, 'Humidity(%)': 20.0,
        'Pressure(in)': 29.95, 'Visibility(mi)': 10.0, 'Wind_Speed(mph)': 2.0,
        'Amenity': 0.0, 'Bump': 0.0, 'Crossing': 0.0, 'Give_Way': 0.0, 'Junction': 0.0,
        'No_Exit': 0.0, 'Railway': 0.0, 'Roundabout': 0.0, 'Station': 0.0, 'Stop': 0.0,
        'Traffic_Calming': 0.0, 'Traffic_Signal': 0.0
    }
])

cal_probs, preds, xgb_p, lgb_p, cat_p = predict_risk(test_cases)

print("\n" + "=" * 75)
print("🏅 INNOVERSE STACKING CHAMPIONSHIP REPORT")
print(f"🎯 Dynamic Optimal Threshold: {best_threshold:.4f}")
print("=" * 75)

titles = [
    "سناریوی ۱: وضعیت بحرانی (طوفان، شب، تقاطع)",
    "سناریوی ۲: وضعیت عادی (روز آفتابی، مسیر استاندارد)"
]

for i in range(len(test_cases)):
    status = "🚨 HIGH RISK (Class 1)" if preds[i] == 1 else "✅ LOW RISK (Class 0)"
    print(f"\n📌 {titles[i]}")
    print(f" ├─ ارزیابی نهایی سیستم: {status}")
    print(f" ├─ احتمال خطر کالیبره‌شده: {cal_probs[i]*100:.2f}%")
    print(f" └─ خروجی مدل‌های پایه: XGB ({xgb_p[i]*100:.1f}%) | LGB ({lgb_p[i]*100:.1f}%) | CAT ({cat_p[i]*100:.1f}%)")

print("\n" + "=" * 75)

# ==========================================
# ۵. راهنمای استفاده روی فایل CSV جدید
# ==========================================
# اگر خواستید یک فایل CSV جدید را ارزیابی کنید، کدهای زیر را از حالت کامنت خارج کنید:
"""
new_data = pd.read_csv('your_test_file.csv')
probs, predictions, _, _, _ = predict_risk(new_data)
new_data['Risk_Probability'] = probs
new_data['Predicted_Risk_Class'] = predictions
new_data.to_csv('predictions_output.csv', index=False)
print("✅ خروجی پیش‌بینی‌ها در فایل predictions_output.csv ذخیره شد.")
"""

In [1]:
import pandas as pd

# ۱. خواندن فایل دیتاست (نام فایل خود را جایگزین کنید)
# اگر فایل شما CSV است از read_csv و اگر اکسل است از read_excel استفاده کنید
df = pd.read_csv('US_Accidents_March23.csv')

# ۲. انتخاب ستون‌های City، County و State
columns_needed = ['City', 'County', 'State']
df_filtered = df[columns_needed]

# ۳. حذف سطر‌های تکراری (Duplicates)
df_unique = df_filtered.drop_duplicates()

# ۴. ذخیره در فایل متنی با جداکننده ویرگول (می‌توانید sep='\t' را برای فاصله‌گذاری با Tab قرار دهید)
df_unique.to_csv('output_cities.txt', index=False, sep=',')

print(f"تعداد سطرهای نهایی بدون تکرار: {len(df_unique)}")

تعداد سطرهای نهایی بدون تکرار: 25094
